# Modelado Dimensional: Esquema Estrella y Copo de Nieve

Objetivo: Entender hechos y dimensiones, diferencias entre estrella y copo de nieve, cuándo desnormalizar y optimizar para analítica.

Contenido compacto para facilitar lectura (secciones cortas + ejemplos).

## 1. Fundamentos Rápidos

Modelado dimensional ≠ modelado relacional tradicional. Se orienta a consulta analítica (OLAP): agregaciones, filtros, drill-down.

Estructura base:
- Tabla de hechos (hechos cuantitativos)
- Tablas de dimensiones (contexto descriptivo)

Principios clave (Kimball):
1. Entender el proceso de negocio (evento medible)
2. Declarar la granularidad (nivel mínimo de detalle)
3. Identificar dimensiones (describen el evento)
4. Identificar hechos (medidas numéricas aditivas)

## 2. Hechos vs Dimensiones

Hecho: Registro medible de un evento (ej: venta).
Dimensión: Punto de vista para filtrar/segmentar (ej: producto, cliente, fecha).

Tipos de hechos:
- Transaccionales (cada evento)
- Acumulados (snapshot)
- Periodos (cambios en intervalos)

Tipos de atributos dimensionales:
- Descriptivos (nombre_producto)
- Jerárquicos (día → mes → año)
- Derivados (segmento_cliente)
- BW Flags (es_promocion)

In [ ]:
# Simulación simple de un proceso de ventas para ilustrar hecho y dimensiones
import pandas as pd, numpy as np
np.random.seed(42)

dim_producto = pd.DataFrame({
    'producto_sk': [101,102,103],
    'producto_id_fuente': ['P-1','P-2','P-3'],
    'nombre_producto': ['Laptop','Mouse','Monitor'],
    'categoria': ['Computo','Accesorios','Computo'],
    'subcategoria': ['Portátiles','Periféricos','Pantallas']
})

dim_cliente = pd.DataFrame({
    'cliente_sk': [201,202,203],
    'cliente_id_fuente': ['C-1','C-2','C-3'],
    'nombre_cliente': ['Ana','Luis','María'],
    'segmento': ['Premium','Estándar','Premium'],
    'pais': ['CO','CO','MX']
})

dim_tiempo = pd.DataFrame({
    'fecha_sk': pd.date_range('2025-01-01','2025-01-05'),
})
dim_tiempo['dia'] = dim_tiempo['fecha_sk'].dt.day
dim_tiempo['mes'] = dim_tiempo['fecha_sk'].dt.month
dim_tiempo['anio'] = dim_tiempo['fecha_sk'].dt.year
dim_tiempo['mes_texto'] = dim_tiempo['fecha_sk'].dt.month.map({1:'Ene'})

# Hechos (granularidad: una línea por producto por cliente por fecha)
hechos_ventas = []
for fecha in dim_tiempo['fecha_sk']:
    for p in dim_producto['producto_sk']:
        for c in dim_cliente['cliente_sk']:
            qty = np.random.poisson(1.2)
            if qty > 0:
                precio = {101:900, 102:25, 103:300}[p]
                hechos_ventas.append([fecha, p, c, qty, qty*precio])

fact_ventas = pd.DataFrame(hechos_ventas, columns=['fecha_sk','producto_sk','cliente_sk','cantidad','importe'])
print('Dim Producto:
', dim_producto)
print('
Dim Cliente:
', dim_cliente)
print('
Dim Tiempo (extracto):
', dim_tiempo.head(3))
print('
Fact Ventas (primeras filas):
', fact_ventas.head())
print('
Granularidad definida: una fila = (fecha, producto, cliente)')

## 3. Esquema Estrella (Star Schema)

Características:
- Dimensiones desnormalizadas (atributos completos)
- Joins simples (fact ↔ dimensión por surrogate key)
- Rendimiento alto en consultas analíticas
- Facilidad de comprensión para usuarios de negocio

Desventaja: Puede haber redundancia en dimensiones amplias (ej: país → región → continente en misma tabla).

## 4. Esquema Copo de Nieve (Snowflake)

Características:
- Dimensiones normalizadas en múltiples tablas
- Menos redundancia, más joins
- Puede reducir almacenamiento y facilitar mantenimiento jerárquico

Desventaja: Consultas más complejas y potencial menor rendimiento.

Uso típico: Dimensiones con jerarquías extensas y relativamente estáticas (geografía, producto complejo).

In [ ]:
# Transformar dim_producto a estructura snowflake (separar categoria y subcategoria)
dim_categoria = (dim_producto[['categoria']]
                .drop_duplicates()
                .reset_index(drop=True))
dim_categoria['categoria_sk'] = dim_categoria.index + 500

dim_subcat = (dim_producto[['subcategoria','categoria']]
              .drop_duplicates()
              .merge(dim_categoria, on='categoria')
              .reset_index(drop=True))
dim_subcat['subcategoria_sk'] = dim_subcat.index + 600

# Nueva dimensión producto normalizada (referencias)
dim_producto_snow = (dim_producto
                     .merge(dim_subcat[['subcategoria','subcategoria_sk']], on='subcategoria')
                     .merge(dim_categoria[['categoria','categoria_sk']], on='categoria')
                     [['producto_sk','nombre_producto','subcategoria_sk','categoria_sk']])

print('Dim Producto (estrella):
', dim_producto)
print('
Dim Producto (snowflake):
', dim_producto_snow)
print('
Dim Subcategoria:
', dim_subcat)
print('
Dim Categoria:
', dim_categoria)

In [ ]:
# Consulta simulada: importe total por categoria vs tiempo (estrella vs copo)
import time
# Estrella
t0 = time.time()
q_estrella = (fact_ventas
              .merge(dim_producto[['producto_sk','categoria']], on='producto_sk')
              .groupby('categoria', as_index=False)['importe'].sum())
t1 = time.time() - t0
# Copo de nieve
t2 = time.time()
q_copo = (fact_ventas
          .merge(dim_producto_snow, on='producto_sk')
          .merge(dim_categoria[['categoria','categoria_sk']], on='categoria_sk')
          .groupby('categoria', as_index=False)['importe'].sum())
t3 = time.time() - t2

print('Resultado Estrella:
', q_estrella)
print('Tiempo Estrella (ms):', round(t1*1000,2))
print('
Resultado Copo de Nieve:
', q_copo)
print('Tiempo Copo (ms):', round(t3*1000,2))
print('
⚠️ Diferencia es mínima porque dataset es pequeño; en grandes volúmenes la reducción de joins importa.')

## 5. Normalización vs Desnormalización

| Aspecto | Normalizado (Snowflake) | Desnormalizado (Estrella) |
|---------|-------------------------|---------------------------|
| Redundancia | Baja | Moderada |
| Rendimiento analítico | Menor (más joins) | Mayor (joins simples) |
| Facilidad de uso | Menor (estructura fragmentada) | Alta (intuitivo) |
| Mantenimiento jerarquías | Más fácil | Puede implicar updates masivos |
| Almacenamiento | Menor | Mayor |

Criterios para elegir desnormalizar:
- Frecuencia alta de consultas agregadas
- Usuarios no técnicos consumen el modelo
- Minimizar tiempos de respuesta en dashboards

Criterios para mantener normalización parcial:
- Dimensiones muy grandes (geografía global detallada)
- Jerarquías complejas con cambios frecuentes
- Optimización de almacenamiento en data warehouse masivo

## 6. Optimización Analítica

Técnicas clave:
1. Surrogate Keys (enteros secuenciales) → Joins más rápidos
2. Columnar Storage (en data warehouse moderno) → Lectura selectiva
3. Particionamiento por fecha o región → Pruning
4. Índices bitmap (algunos motores) → Filtros rápidos en baja cardinalidad
5. Pre-Agregaciones / Tablas resumen → Menos cómputo en tiempo real
6. Caché en motores MPP (BigQuery, Redshift) → Reutilización plan/resultado
7. Ordenar físicamente por surrogate keys y fecha para compresión y scans eficientes.

Evitar: Desnormalizar exageradamente creando dimensiones gigantes de 1..N atributos no usados; sobre-indexar tablas de hechos.

## 7. Ejercicio Guiado

Escenario: Marketplace con ventas multi-país y campañas de marketing.

Tareas:
1. Define granularidad para fact_ventas_marketing (¿por evento de clic? ¿por transacción?)
2. Lista dimensiones necesarias (producto, cliente, fecha, campaña, canal, país).
3. Clasifica cuáles podrían normalizarse en snowflake (ej: país → región → continente).
4. Especifica tres métricas (hechos) aditivas.
5. Propón un resumen mensual pre-agregado.

Completa en la celda siguiente.

In [ ]:
# Plantilla de respuesta del ejercicio
plantilla = '''
# Ejercicio: Diseño Modelo Dimensional Marketplace

1. Granularidad: [Describe aquí]
2. Dimensiones: [Lista]
3. Jerarquías a snowflake: [Lista]
4. Métricas aditivas: [Lista y justificación de aditividad]
5. Tabla resumen mensual: [Columnas y propósito]

Riesgos si se desnormaliza demasiado: [Explica]
Compensaciones rendimiento vs mantenimiento: [Explica]
'''
print(plantilla)

## 8. Checklist Rápido para tu Diseño

- [ ] Granularidad clara y documentada
- [ ] Hechos son numéricos y aditivos (o semi-aditivos documentados)
- [ ] Dimensiones contienen atributos útiles de filtrado
- [ ] Claves sustitutas (surrogate) implementadas
- [ ] Jerarquías definidas (fecha, producto, geografía)
- [ ] Estrategia de slowly changing dimensions (SCD) considerada (se verá más adelante)
- [ ] Plan de índices / particiones alineado a patrones de consulta
- [ ] Justificación de cualquier desnormalización
- [ ] Tabla(s) de agregación si consultas repetitivas pesadas

## 9. Resumen

- Hechos registran eventos y contienen métricas.
- Dimensiones describen el contexto y soportan cortes analíticos.
- Estrella: simplicidad y velocidad; Copo: ahorro y estructura jerárquica.
- Desnormalizar es una decisión consciente para acelerar analítica.
- Optimizar implica balancear: rendimiento, claridad, mantenimiento.

Siguiente tema (futuro): Slowly Changing Dimensions (SCD) y manejo de historiales.

---
**Referencias Breves**
- Kimball, R. *The Data Warehouse Toolkit*.
- Inmon, W. *Building the Data Warehouse*.
- Corr, L. *Agile Data Warehouse Design*.
- Gupta, A. *Database System Concepts* (capítulo data warehousing).